# DistilBERT caption classifier

Fine-tune DistilBERT on Molmo captions for species classification (text-only baseline). Set `variant` to `"mini"` (FungiTastic-M) or `"full"` (FungiTastic).

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
import evaluate

from vlm_utils import load_captions, load_split_df

# --- configuration ---
dataset_dir = '/FungiTastic'
variant = 'mini'  # 'mini' | 'full'
caption_dir = f'{dataset_dir}/captions/{variant}'  # folder with <filename>.json
output_dir = f'checkpoints/distilbert-ft-{variant}'
results_dir = 'results'

df_train = load_split_df(dataset_dir, variant, 'train')
df_val = load_split_df(dataset_dir, variant, 'val')
df_test = load_split_df(dataset_dir, variant, 'test')

captions_train = load_captions(caption_dir, df_train)
captions_val = load_captions(caption_dir, df_val)
captions_test = load_captions(caption_dir, df_test)

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length')

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

dataset_train = Dataset.from_dict({'label': df_train['category_id'].values, 'text': captions_train}).map(tokenize, batched=True)
dataset_val = Dataset.from_dict({'label': df_val['category_id'].values, 'text': captions_val}).map(tokenize, batched=True)
dataset_test = Dataset.from_dict({'label': df_test['category_id'].values, 'text': captions_test}).map(tokenize, batched=True)

num_labels = int(df_train['category_id'].nunique())
print(f'{variant}: {len(df_train)} train, {len(df_val)} val, {len(df_test)} test, {num_labels} classes')

## Train

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    compute_metrics=compute_metrics,
)

trainer.train()

## Export test logits

Saves raw logits for `fusion.ipynb` (Table 7 evaluation on the test split).

In [ ]:
from pathlib import Path

Path(results_dir).mkdir(parents=True, exist_ok=True)

checkpoints = sorted(Path(output_dir).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
assert checkpoints, f'No checkpoints in {output_dir}'
best_ckpt = str(checkpoints[-1])
print('Loading', best_ckpt)

model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)
trainer_eval = Trainer(model=model)

preds = trainer_eval.predict(dataset_test)
path = f'{results_dir}/bert_logits_{variant}_test.pth'
torch.save(torch.tensor(preds.predictions), path)
print('Saved', path)